# Links

GitHub repository:
https://github.com/nylandgit/IND320-Repo

Streamlit app:
https://nylandgit-ind320-repo.streamlit.app/

Code runthrough video:


# References

In the interest of avoiding claims of self-plagiarism (don't know if this is necessary for IND320?):

Bulks of the following code is copied and adapted from Thor Martin Nyland's Project Work Part 1. Parts of the code is copied directly without any modification. The AI Statement is copied directly, as there has been no change in AI usage since Part 1.

Parts of the code have been copied from khliland's IND320 GitHub Repository. These parts are highlighted where applicable.

# AI Statement and Descriptive Log

## AI Statement

This project has used the free VS Code Copilot generative AI to: Generate code, identify bugs, fix bugs, brainstorm technical solutions, and to teach the process of coding through in-depth explanations of each generated line of code. Some Google searches have also involved Google's built-in generative AI.

The project author has steered the AI's coding through accepting, rejecting, and modifying the AI's code based on the author's intuition. The author has requested code explanations, studied the AI's code, and made explanatory comments for each code block based on the author's understanding.

The code provided is therefore created partly by the author and partly by generative AI. The explanatory code comments are written exclusively by the author.

## Descriptive Log

### Text
* Text

<p></p>
<p></p>

### Text
* Text


# Non-Streamlit Code

## 1. Atlas NVE

### 1.1 WIP Precipitation Area Sampling

In [21]:
# This code has been copied and adapted from Kristian Hovde Lilands IND320
# lecture held at NMBU on 26.10.2026.
import geopandas as gpd

# Read file
gdf_precip = gpd.read_file("../shapefiles/precipitation areas/NVE_Region.shp")

# Generate random points
n = 1000
points_precip = gdf_precip['geometry'].sample_points(size=n, random_state=42)

# Transform WGS84 (EPSG:4326) for latitude and longitude (?)
points_wgs84_precip = points_precip.to_crs(epsg=4326)

# Extract coordinates and print
for idx_precip, multipoint_precip in points_wgs84_precip.items():
    region_name_precip = gdf_precip.loc[idx_precip, 'REGION']
    #print(f"Region: {region_name_precip}")

    for point_precip in multipoint_precip.geoms:
        # Do something more useful with the coordinates than this (?)
        lat_precip = point_precip.y
        lon_precip = point_precip.x

print(lat_precip)


59.88429920570754


### 1.2 WIP Electricity Areas Sampling

In [40]:
# This code has been copied and adapted from Kristian Hovde Lilands IND320
# lecture held at NMBU on 26.10.2026.

# Read file
gdf_el = gpd.read_file("../shapefiles/electricity areas/Budområder.shp")

# Generate random points
n = 1000
points_el = gdf_el['geometry'].sample_points(size=n, random_state=42)

# Transform WGS84 (EPSG:4326) for latitude and longitude (?)
points_wgs84_el = points_el.to_crs(epsg=4326)

# Extract coordinates and save them
locations_by_region_el = {}
for idx_el, multipoint_el in points_wgs84_el.items():
    region_name_el = gdf_el.loc[idx_el, 'budomr']
    print(f"Region: {region_name_el}")

    for point_el in multipoint_el.geoms:
        # Do something more useful with the coordinates than this (?)
        lat_el = point_el.y
        lon_el = point_el.x
        locations_by_region_el.setdefault(region_name_el, []).append((lat_el, lon_el))

print(lat_el)
print(gdf_el.head())


Region: NO 1
Region: NO 2
Region: NO 3
Region: NO 4
Region: NO 5
60.52098299239371
   objectid budomr  globalid    st_area(sh    st_perimet  \
0       1.0   NO 1       NaN  6.052902e+10  1.657761e+06   
1       2.0   NO 2       NaN  6.875969e+10  1.256473e+06   
2       3.0   NO 3       NaN  1.042199e+11  1.773323e+06   
3       4.0   NO 4       NaN  2.032500e+11  3.385499e+06   
4       5.0   NO 5       NaN  3.334870e+10  1.106762e+06   

                                            geometry  
0  POLYGON ((337473.63 6970542.97, 337711.9 69703...  
1  POLYGON ((-9833.91 6726499.86, -9836 6726404.0...  
2  POLYGON ((308588.7 7261519.43, 313576.63 72590...  
3  POLYGON ((973920.93 7962739.19, 974922.77 7962...  
4  POLYGON ((101466.43 6881318.54, 101525.43 6881...  


### 1.3 WIP Open-Meteo API

In [ ]:
import pandas as pd
import requests

def download_region(region, start_date, end_date):
    locations = locations_by_region_el[region]
    url = "https://archive-api.open-meteo.com/v1/archive"
    results = []

    for start in range(0, len(locations), 50):
        batch = locations[start:start + 50]

        params = {
            "latitude": ",".join(str(lat) for lat, lon in batch),
            "longitude": ",".join(str(lon) for lat, lon in batch),
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "temperature_2m,precipitation",
            "models": "era5",
            "timezone": "UTC",
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        api_results = response.json()

        if isinstance(api_results, dict):
            api_results = [api_results]

        if len(api_results) != len(batch):
            raise ValueError("The API returned an unexpected number of locations.")

        for (lat, lon), location_data in zip(batch, api_results):
            hourly = pd.DataFrame(location_data["hourly"])
            hourly["region"] = region
            hourly["latitude"] = lat
            hourly["longitude"] = lon
            results.append(hourly)

    return pd.concat(results, ignore_index=True)

download_region("NO 3", "2024-01-01", "2024-01-31")


### 1.3 WIP alternative

In [43]:
import hashlib
import json
import time
from pathlib import Path

import pandas as pd
import requests


def download_region(region, start_date, end_date):
    locations = locations_by_region_el[region]
    url = "https://archive-api.open-meteo.com/v1/archive"
    cache_dir = Path("open_meteo_cache")
    cache_dir.mkdir(exist_ok=True)
    results = []

    for start in range(0, len(locations), 50):
        batch = locations[start:start + 50]

        params = {
            "latitude": ",".join(str(lat) for lat, lon in batch),
            "longitude": ",".join(str(lon) for lat, lon in batch),
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "temperature_2m,precipitation",
            "models": "era5",
            "timezone": "UTC",
        }

        # Make a unique cache filename for this batch and request.
        cache_key = hashlib.sha256(
            json.dumps(params, sort_keys=True).encode()
        ).hexdigest()
        cache_file = cache_dir / f"{cache_key}.json"

        if cache_file.exists():
            with cache_file.open() as file:
                api_results = json.load(file)
        else:
            response = requests.get(url, params=params)
            response.raise_for_status()
            api_results = response.json()

            # Save only successful responses.
            with cache_file.open("w") as file:
                json.dump(api_results, file)

            time.sleep(1)

        if isinstance(api_results, dict):
            api_results = [api_results]

        if len(api_results) != len(batch):
            raise ValueError("The API returned an unexpected number of locations.")

        for (lat, lon), location_data in zip(batch, api_results):
            hourly = pd.DataFrame(location_data["hourly"])
            hourly["region"] = region
            hourly["latitude"] = lat
            hourly["longitude"] = lon
            results.append(hourly)

    return pd.concat(results, ignore_index=True)

no3_2024 = download_region("NO 3", "2024-01-01", "2024-01-31")


In [ ]:
# Finding BRZ pairings to get total in/out Norway (used map on Transparency
# Platform website to find pairings).
brz_pairings = {
    "NO_1": ["SE_3"],
    "NO_2": ["DK_1", "GB", "DE_LU", "NL"],
    "NO_3": ["SE_2"],
    "NO_4": ["SE_1", "SE_2", "FI", "RU"],
}
